# FREUID 2026: A100 newest-dataset inference test

This notebook runs the frozen best-public ConvNeXtV2 Large checkpoint on the newest
FREUID competition archive. It benchmarks real image decoding and inference on one
A100 40 GB, selects the fastest stable batch size, writes the complete submission,
and bundles reproducibility evidence for direct browser download.

Competition objective: detect identity-document fraud across physical manipulation,
digital/GenAI edits, and print-and-capture attacks. The output is a fraud probability,
where larger values indicate greater likelihood of fraud.

Official references:

- [Competition](https://www.kaggle.com/competitions/the-freuid-challenge-2026-ijcai-ecai/)
- [Reproducibility thread](https://www.kaggle.com/competitions/the-freuid-challenge-2026-ijcai-ecai/discussion/718637)
- [Leaderboard caveat](https://www.kaggle.com/competitions/the-freuid-challenge-2026-ijcai-ecai/discussion/723604)
- [A100 runtime requirement](https://www.kaggle.com/competitions/the-freuid-challenge-2026-ijcai-ecai/discussion/723556)

This notebook performs inference only. It does not train, alter the checkpoint,
tune a decision threshold, add TTA, or insert dummy predictions.


## 1. Install the frozen inference dependencies

Restarting the runtime after this cell is not required. Package installation is kept
separate so setup time is not included in the inference benchmark.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "kagglehub==1.0.2", "gdown==5.2.0",
    "timm==1.0.15", "Pillow==10.4.0",
])
from importlib.metadata import version

kagglehub_version = version("kagglehub")
if kagglehub_version != "1.0.2":
    raise RuntimeError(f"Expected kagglehub 1.0.2, got {kagglehub_version}.")
print(f"Dependencies installed. kagglehub {kagglehub_version}")


## 2. Configuration

`FORCE_BATCH_SIZE=None` runs the benchmark and automatically chooses the fastest
successful candidate. After the first run, set it to a known-good value to skip the
sweep. Keep all model and preprocessing settings unchanged.


In [ ]:
import gc
import getpass
import hashlib
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

import gdown
import kagglehub
import numpy as np
import pandas as pd
import PIL
import timm
import torch
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

COMPETITION = "the-freuid-challenge-2026-ijcai-ecai"
EXPECTED_ROWS = 142_818

MODEL_NAME = "convnextv2_large.fcmae_ft_in22k_in1k_384"
IMAGE_SIZE = 672
EXPECTED_EPOCH = 3
PUBLIC_CHECKPOINT_URL = (
    "https://drive.google.com/uc?id=1PGhqwLlZHwfebCOoNJO-XqsjYkwD7Ip1"
)
EXTRACT_DIR = Path("/content/freuid_latest")
LOCAL_RUN_DIR = Path("/content/freuid_a100_run")

BATCH_CANDIDATES = [8, 12, 16, 20, 24, 28, 32]
FORCE_BATCH_SIZE = None
BENCHMARK_IMAGES = 1_024
CPU_LIMIT = 24
NUM_WORKERS = min(16, os.cpu_count() or 1, CPU_LIMIT)
PREFETCH_FACTOR = 4
MIN_FREE_DISK_GB = 45

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

for path in (EXTRACT_DIR, LOCAL_RUN_DIR):
    path.mkdir(parents=True, exist_ok=True)

print({
    "batch_candidates": BATCH_CANDIDATES,
    "force_batch_size": FORCE_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "local_output": str(LOCAL_RUN_DIR),
})


## 3. Verify the organizer-equivalent accelerator

The organizers specified one A100 40 GB, up to 24 CPUs, and a six-hour inference
limit. The cell stops on a different GPU so the timing report is not mislabeled.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select an A100 GPU runtime.")
if torch.cuda.device_count() != 1:
    raise RuntimeError(f"Expected exactly one GPU, found {torch.cuda.device_count()}.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_props = torch.cuda.get_device_properties(0)
gpu_memory_gb = gpu_props.total_memory / 1024**3
cpu_count = os.cpu_count() or 1

if "A100" not in gpu_name.upper():
    raise RuntimeError(f"Expected an A100, found {gpu_name}.")
if not 38.0 <= gpu_memory_gb <= 45.0:
    raise RuntimeError(
        f"Expected the organizer-equivalent A100 40 GB, found {gpu_memory_gb:.1f} GB."
    )

torch.set_num_threads(min(cpu_count, CPU_LIMIT))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

subprocess.run(["nvidia-smi"], check=True)
print({"gpu": gpu_name, "gpu_memory_gb": round(gpu_memory_gb, 2), "cpus": cpu_count})


## 4. Authenticate and download the newest competition archive

The cell first reads the Colab Secret named `KAGGLE_API_TOKEN`. In Colab, enable notebook
access for that secret using the key icon in the left sidebar. It only displays the token
source, never the value. If the secret is unavailable, a hidden paste prompt is used.
The organizer is currently uploading and indexing the private test release. KaggleHub's
bulk call is the correct efficient route once that process completes. A temporary 404 is
reported clearly so the cell can be retried without spending hours on per-image requests.


In [ ]:
kaggle_token = os.environ.get("KAGGLE_API_TOKEN", "").strip()
token_source = "environment variable"

if not kaggle_token:
    try:
        from google.colab import userdata
        kaggle_token = (userdata.get("KAGGLE_API_TOKEN") or "").strip()
        token_source = "Colab Secrets"
    except Exception as exc:
        print(f"Colab Secret unavailable ({type(exc).__name__}); using hidden prompt.")

if not kaggle_token:
    kaggle_token = getpass.getpass("Paste KAGGLE_API_TOKEN and press Enter: ").strip()
    token_source = "hidden prompt"

if not kaggle_token:
    raise ValueError("KAGGLE_API_TOKEN cannot be empty.")

os.environ["KAGGLE_API_TOKEN"] = kaggle_token
print(f"Kaggle token loaded from {token_source}; value remains hidden.")
print("Kaggle token configured for KaggleHub. Checking bulk-archive readiness...")

disk_usage = shutil.disk_usage("/content")
free_disk_gb = disk_usage.free / 1024**3
print(f"Free local disk before download: {free_disk_gb:.2f} GB")
if free_disk_gb < MIN_FREE_DISK_GB:
    raise RuntimeError(
        f"Only {free_disk_gb:.2f} GB is free. At least {MIN_FREE_DISK_GB} GB is required "
        "for the archive, extracted images, checkpoint, and outputs."
    )

download_started = time.perf_counter()
try:
    competition_path = Path(kagglehub.competition_download(
        COMPETITION,
        output_dir=str(EXTRACT_DIR),
        force_download=True,
    ))
except Exception as exc:
    if "404" in str(exc):
        raise RuntimeError(
            "The private-test bulk archive is not ready on Kaggle yet. The organizers "
            "said the upload/indexing process is still in progress. Retry this cell after "
            "they confirm completion; do not download 134,997 images individually."
        ) from exc
    raise RuntimeError(
        f"KaggleHub competition download failed: {type(exc).__name__}: {exc}"
    ) from exc
download_seconds = time.perf_counter() - download_started

if not competition_path.exists() or not competition_path.is_dir():
    raise RuntimeError(f"KaggleHub returned an invalid competition path: {competition_path}")
print({
    "competition_path": str(competition_path),
    "kagglehub_version": kagglehub.__version__,
    "download_minutes": round(download_seconds / 60, 2),
})


## 5. Enforce complete test-image coverage

KaggleHub has already extracted the archive. The sample submission is the exact ID
contract. The notebook indexes images recursively,
but only accepts IDs listed in `sample_submission.csv`. Missing or duplicate images stop
the run; no fallback score is allowed.


In [ ]:
sample_candidates = sorted(competition_path.rglob("sample_submission.csv"))
if len(sample_candidates) != 1:
    raise RuntimeError(
        f"Expected one sample_submission.csv, found {len(sample_candidates)}: {sample_candidates}"
    )
sample_path = sample_candidates[0]
sample = pd.read_csv(sample_path, dtype={"id": str})

if list(sample.columns) != ["id", "label"]:
    raise RuntimeError(f"Unexpected submission columns: {list(sample.columns)}")
if len(sample) != EXPECTED_ROWS:
    raise RuntimeError(f"Expected {EXPECTED_ROWS:,} sample rows, found {len(sample):,}.")
if sample["id"].duplicated().any():
    raise RuntimeError("sample_submission.csv contains duplicate IDs.")

required_ids = set(sample["id"])
image_by_id = {}
duplicate_images = {}

scan_started = time.perf_counter()
for root, _, filenames in os.walk(competition_path):
    root_path = Path(root)
    for filename in filenames:
        path = root_path / filename
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        image_id = path.stem
        if image_id not in required_ids:
            continue
        if image_id in image_by_id:
            duplicate_images.setdefault(image_id, [image_by_id[image_id]]).append(path)
        else:
            image_by_id[image_id] = path
scan_seconds = time.perf_counter() - scan_started

missing_ids = [image_id for image_id in sample["id"] if image_id not in image_by_id]
if duplicate_images:
    preview = {key: [str(p) for p in value] for key, value in list(duplicate_images.items())[:5]}
    raise RuntimeError(f"Duplicate test-image IDs found: {preview}")
if missing_ids:
    raise RuntimeError(
        f"Newest archive is still missing {len(missing_ids):,} of {EXPECTED_ROWS:,} images. "
        f"First missing IDs: {missing_ids[:10]}. Do not create a dummy-filled submission."
    )

ordered_image_paths = [image_by_id[image_id] for image_id in sample["id"]]
parent_counts = pd.Series([str(path.parent.relative_to(competition_path)) for path in ordered_image_paths]).value_counts()
print({
    "sample_rows": len(sample),
    "resolved_images": len(ordered_image_paths),
    "scan_minutes": round(scan_seconds / 60, 2),
})
display(parent_counts.rename("images").to_frame())


## 6. Resolve and validate the frozen checkpoint

The checkpoint is downloaded from the exact public Google Drive file used by the Docker
build. Google Drive is not mounted and no personal Drive permission is required. The
checkpoint identity and model metadata are recorded before inference.


In [ ]:
checkpoint_path = Path("/content/freuid_best_epoch03.pt")
if not checkpoint_path.is_file():
    result = gdown.download(
        url=PUBLIC_CHECKPOINT_URL,
        output=str(checkpoint_path),
        quiet=False,
        fuzzy=True,
    )
    if not result:
        raise RuntimeError("Public checkpoint download failed.")

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_sha256 = sha256_file(checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

actual_model_name = checkpoint.get("model_name") or checkpoint.get("config", {}).get("model_name")
actual_image_size = int(checkpoint.get("image_size") or checkpoint.get("config", {}).get("image_size", 0))
actual_epoch = int(checkpoint.get("epoch", -1))

if actual_model_name != MODEL_NAME:
    raise RuntimeError(f"Checkpoint model mismatch: {actual_model_name!r} != {MODEL_NAME!r}")
if actual_image_size != IMAGE_SIZE:
    raise RuntimeError(f"Checkpoint image-size mismatch: {actual_image_size} != {IMAGE_SIZE}")
if actual_epoch != EXPECTED_EPOCH:
    raise RuntimeError(f"Checkpoint epoch mismatch: {actual_epoch} != {EXPECTED_EPOCH}")

print({
    "checkpoint": str(checkpoint_path),
    "checkpoint_gb": round(checkpoint_path.stat().st_size / 1024**3, 3),
    "checkpoint_sha256": checkpoint_sha256,
    "model_name": actual_model_name,
    "image_size": actual_image_size,
    "epoch": actual_epoch,
    "best_loss": checkpoint.get("best_loss"),
})


## 7. Build the exact frozen inference pipeline

Preprocessing matches the packaged Docker entrypoint: RGB conversion, bicubic resize of
the shorter side, center crop, tensor conversion, and ImageNet normalization. A100 uses
BF16 autocast and channels-last tensors for throughput.


In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

class FreuidInferenceDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = list(image_paths)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        path = self.image_paths[index]
        with Image.open(path) as image:
            tensor = self.transform(image.convert("RGB"))
        return index, tensor

dataset = FreuidInferenceDataset(ordered_image_paths, eval_transform)
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=1)
load_result = model.load_state_dict(checkpoint["model"], strict=True)
model = model.to(device, memory_format=torch.channels_last).eval()
del checkpoint
gc.collect()
torch.cuda.empty_cache()

amp_dtype = torch.bfloat16
print(load_result)
print(f"Dataset rows: {len(dataset):,}; AMP dtype: {amp_dtype}")


## 8. Benchmark batch sizes on real images

This includes image decoding, preprocessing, host-to-device transfer, and model inference.
An out-of-memory candidate is recorded and skipped. Selection is based on measured images
per second, not simply the largest batch.


In [ ]:
def make_loader(source_dataset, batch_size):
    kwargs = dict(
        dataset=source_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

warmup_loader = make_loader(Subset(dataset, range(min(64, len(dataset)))), min(BATCH_CANDIDATES))
with torch.inference_mode():
    for _, images in warmup_loader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            _ = model(images)
torch.cuda.synchronize()
del warmup_loader

benchmark_count = min(BENCHMARK_IMAGES, len(dataset))
benchmark_subset = Subset(dataset, range(benchmark_count))
benchmark_rows = []

candidates = [FORCE_BATCH_SIZE] if FORCE_BATCH_SIZE else BATCH_CANDIDATES
for batch_size in candidates:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    loader = make_loader(benchmark_subset, batch_size)
    started = time.perf_counter()
    status = "ok"
    error = ""
    processed = 0
    try:
        with torch.inference_mode():
            for _, images in loader:
                images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    _ = model(images)
                processed += len(images)
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError as exc:
        status = "oom"
        error = str(exc).splitlines()[0]
        if "images" in locals():
            del images
        if "_" in locals():
            del _
        gc.collect()
        torch.cuda.empty_cache()
    elapsed = time.perf_counter() - started
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    images_per_second = processed / elapsed if status == "ok" else 0.0
    row = {
        "batch_size": batch_size,
        "status": status,
        "images": processed,
        "seconds": elapsed,
        "images_per_second": images_per_second,
        "peak_allocated_gb": peak_gb,
        "error": error,
    }
    benchmark_rows.append(row)
    print(json.dumps(row, default=str))
    del loader

benchmark_df = pd.DataFrame(benchmark_rows)
successful = benchmark_df[benchmark_df["status"] == "ok"]
if successful.empty:
    raise RuntimeError("Every batch-size candidate failed.")

selected_batch_size = int(successful.sort_values("images_per_second", ascending=False).iloc[0]["batch_size"])
benchmark_path = LOCAL_RUN_DIR / "a100_batch_benchmark.csv"
benchmark_df.to_csv(benchmark_path, index=False)
display(benchmark_df)
print(f"Selected batch size: {selected_batch_size}")


## 9. Run complete inference

The timer includes DataLoader startup, decoding, preprocessing, transfers, and inference,
but excludes download, extraction, model loading, and the batch-size sweep. Progress records
include throughput, ETA, and CUDA memory.


In [ ]:
full_loader = make_loader(dataset, selected_batch_size)
predictions = np.empty(len(dataset), dtype=np.float32)
progress_log = []

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
inference_started = time.perf_counter()
processed = 0

with torch.inference_mode():
    for step, (indices, images) in enumerate(full_loader, start=1):
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits = model(images).flatten()
            probabilities = torch.sigmoid(logits)
        predictions[indices.numpy()] = probabilities.float().cpu().numpy()
        processed += len(images)

        if step == 1 or step % 100 == 0 or processed == len(dataset):
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - inference_started
            rate = processed / elapsed
            eta_seconds = (len(dataset) - processed) / rate
            record = {
                "step": step,
                "processed": processed,
                "total": len(dataset),
                "images_per_second": round(rate, 3),
                "elapsed_minutes": round(elapsed / 60, 3),
                "eta_minutes": round(eta_seconds / 60, 3),
                "allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 3),
                "reserved_gb": round(torch.cuda.memory_reserved() / 1024**3, 3),
            }
            progress_log.append(record)
            print(json.dumps(record))

torch.cuda.synchronize()
inference_seconds = time.perf_counter() - inference_started
peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
inference_images_per_second = len(dataset) / inference_seconds

print({
    "inference_minutes": round(inference_seconds / 60, 3),
    "images_per_second": round(inference_images_per_second, 3),
    "peak_allocated_gb": round(peak_memory_gb, 3),
    "under_six_hours": inference_seconds < 6 * 60 * 60,
})


## 10. Validate and save the submission and evidence

Validation checks exact row order and count, finite probabilities in `[0, 1]`, and the
six-hour runtime requirement. The validated files are bundled into one ZIP for direct
browser download; Google Drive is not mounted.


In [ ]:
submission = sample.copy()
submission["label"] = predictions

if len(submission) != EXPECTED_ROWS:
    raise RuntimeError(f"Output has {len(submission):,} rows, expected {EXPECTED_ROWS:,}.")
if not submission["id"].equals(sample["id"]):
    raise RuntimeError("Output IDs or row order differ from sample_submission.csv.")
if submission["id"].duplicated().any():
    raise RuntimeError("Output contains duplicate IDs.")
if not np.isfinite(submission["label"].to_numpy()).all():
    raise RuntimeError("Output contains NaN or infinite predictions.")
if not submission["label"].between(0.0, 1.0).all():
    raise RuntimeError("Output predictions are outside [0, 1].")
if inference_seconds >= 6 * 60 * 60:
    raise RuntimeError(f"Inference took {inference_seconds / 3600:.3f} hours, exceeding six hours.")

submission_path = LOCAL_RUN_DIR / "submission_convnextv2_epoch03_newest_a100.csv"
progress_path = LOCAL_RUN_DIR / "inference_progress.json"
manifest_path = LOCAL_RUN_DIR / "a100_run_manifest.json"

submission.to_csv(submission_path, index=False, float_format="%.8g")
progress_path.write_text(json.dumps(progress_log, indent=2), encoding="utf-8")

manifest = {
    "competition": COMPETITION,
    "expected_rows": EXPECTED_ROWS,
    "actual_rows": len(submission),
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "checkpoint_epoch": EXPECTED_EPOCH,
    "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_path": str(checkpoint_path),
    "sample_submission_path": str(sample_path),
    "gpu": gpu_name,
    "gpu_memory_gb": gpu_memory_gb,
    "visible_gpu_count": torch.cuda.device_count(),
    "cpu_count": cpu_count,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "selected_batch_size": selected_batch_size,
    "amp_dtype": str(amp_dtype),
    "inference_seconds": inference_seconds,
    "inference_images_per_second": inference_images_per_second,
    "peak_allocated_gb": peak_memory_gb,
    "under_six_hours": inference_seconds < 6 * 60 * 60,
    "prediction_min": float(submission["label"].min()),
    "prediction_max": float(submission["label"].max()),
    "prediction_mean": float(submission["label"].mean()),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "timm_version": timm.__version__,
    "pillow_version": PIL.__version__,
    "kagglehub_version": kagglehub.__version__,
    "submission_sha256": sha256_file(submission_path),
    "benchmark": benchmark_rows,
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

bundle_base = Path("/content/freuid_a100_repro_test")
bundle_path = Path(shutil.make_archive(str(bundle_base), "zip", root_dir=LOCAL_RUN_DIR))

print(json.dumps({
    "status": "PASS",
    "selected_batch_size": selected_batch_size,
    "num_workers": NUM_WORKERS,
    "inference_minutes": round(inference_seconds / 60, 3),
    "images_per_second": round(inference_images_per_second, 3),
    "peak_allocated_gb": round(peak_memory_gb, 3),
    "submission_sha256": manifest["submission_sha256"],
    "local_files": [
        str(submission_path), str(benchmark_path), str(progress_path), str(manifest_path)
    ],
    "download_bundle": str(bundle_path),
}, indent=2))

from google.colab import files
files.download(str(bundle_path))


## Result to report

Send back the final `PASS` JSON block plus `a100_batch_benchmark.csv`. Those contain the
selected batch size, worker count, full inference time, throughput, and peak A100 memory.
The Kaggle-ready CSV is `submission_convnextv2_epoch03_newest_a100.csv` in
`/content/freuid_a100_run/`. The final cell also downloads
`freuid_a100_repro_test.zip` directly through the browser.
